# Orquestador Bronze → Silver
Generado con la skill `notebooks-medallion` desde `metadata/`. **No lo edites a mano.**

Ejecuta 69 notebooks con `notebookutils.notebook.runMultiple` como un DAG, con el máximo paralelismo que permiten las dependencias.
Al final escribe una fila en `control.dbo.log_orquestacion`. Cada notebook escribe la suya en `control.dbo.log_ejecucion` con el mismo `id_orquestacion`.

In [ ]:
proyecto = "GreatAdventures"
ws_control = "GreatAdventuresControl"
lh_control = "control"
concurrencia = 0           # 0 = máximo: tantas como actividades (la sesión de Spark puede limitarlo)
solo = ""                  # lista separada por comas para ejecutar solo esas actividades (y sus dependencias)
simular = False            # se pasa a cada notebook
modo = "auto"              # auto | reconstruir, se pasa a cada notebook
timeout_celda_s = 3600
timeout_total_s = 43200
reintentos = 1

In [ ]:
# Utilidades comunes (iguales en todos los notebooks generados): rutas, log en el lakehouse de control y MERGE.
import json, uuid, datetime as dt
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def ruta_lakehouse(ws, lh):
    return f"abfss://{ws}@{ONELAKE}/{lh}.Lakehouse"

RUTA_CONTROL = f"{ruta_lakehouse(ws_control, lh_control)}/Tables/dbo"
ID_EJECUCION = str(uuid.uuid4())
INICIO = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

def contexto(clave, defecto=None):
    try:
        return notebookutils.runtime.context.get(clave) or defecto
    except Exception:
        return defecto

ESQUEMA_LOG_EJECUCION = T.StructType([
    T.StructField("id_ejecucion", T.StringType()),
    T.StructField("id_orquestacion", T.StringType()),
    T.StructField("proyecto", T.StringType()),
    T.StructField("capa", T.StringType()),
    T.StructField("objeto", T.StringType()),
    T.StructField("destino", T.StringType()),
    T.StructField("tipo_carga", T.StringType()),
    T.StructField("estado", T.StringType()),
    T.StructField("simulado", T.BooleanType()),
    T.StructField("inicio", T.TimestampType()),
    T.StructField("fin", T.TimestampType()),
    T.StructField("duracion_s", T.DoubleType()),
    T.StructField("ficheros_leidos", T.LongType()),
    T.StructField("filas_leidas", T.LongType()),
    T.StructField("filas_unicas", T.LongType()),
    T.StructField("filas_duplicadas", T.LongType()),
    T.StructField("filas_borrado_origen", T.LongType()),
    T.StructField("filas_insertadas", T.LongType()),
    T.StructField("filas_actualizadas", T.LongType()),
    T.StructField("filas_borradas", T.LongType()),
    T.StructField("marca_desde", T.StringType()),
    T.StructField("marca_hasta", T.StringType()),
    T.StructField("error", T.StringType()),
    T.StructField("notebook", T.StringType()),
    T.StructField("workspace", T.StringType()),
    T.StructField("usuario", T.StringType()),
    T.StructField("spark_app_id", T.StringType()),
    T.StructField("parametros", T.StringType()),
])

def registrar_ejecucion(**valores):
    """Añade una fila a control.dbo.log_ejecucion. Solo append: no hay conflictos entre notebooks en paralelo."""
    fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    base = dict(id_ejecucion=ID_EJECUCION, proyecto=proyecto, inicio=INICIO, fin=fin,
                duracion_s=(fin - INICIO).total_seconds(), notebook=contexto("currentNotebookName"),
                workspace=contexto("currentWorkspaceName"), usuario=contexto("userName"),
                spark_app_id=spark.sparkContext.applicationId)
    base.update(valores)
    fila = tuple(base.get(c.name) for c in ESQUEMA_LOG_EJECUCION)
    (spark.createDataFrame([fila], ESQUEMA_LOG_EJECUCION)
         .write.format("delta").mode("append").option("mergeSchema", "true")
         .save(f"{RUTA_CONTROL}/log_ejecucion"))

def es_tabla_delta(ruta):
    try:
        return DeltaTable.isDeltaTable(spark, ruta)
    except Exception:
        return False

def con_hash(df, excluir):
    """Añade _hash_fila: sha2 de las columnas de negocio. Sirve para no reescribir filas que no han cambiado."""
    cols = [c for c in df.columns if c not in excluir and not c.startswith("_")]
    return df.withColumn("_hash_fila", F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000")) for c in sorted(cols)]), 256))

def anadir_columnas_nuevas(ruta, df, evolucion):
    """Evolución de esquema controlada: añade al destino las columnas nuevas del origen o falla si no está permitido."""
    destino = spark.read.format("delta").load(ruta)
    nuevas = [f for f in df.schema.fields if f.name not in destino.columns]
    if not nuevas:
        return []
    if not evolucion:
        raise ValueError(f"Columnas nuevas en origen y evolucion_esquema=False: {[f.name for f in nuevas]}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in nuevas)
    spark.sql(f"ALTER TABLE delta.`{ruta}` ADD COLUMNS ({cols})")
    return [f.name for f in nuevas]

def columnas_con_tipo_cambiado(ruta, df):
    """Columnas que existen en destino con un tipo distinto al del origen."""
    actual = {f.name: f.dataType for f in spark.read.format("delta").load(ruta).schema.fields}
    return [f"{f.name}: {actual[f.name].simpleString()} -> {f.dataType.simpleString()}"
            for f in df.schema.fields if f.name in actual and actual[f.name] != f.dataType]

def fusionar(ruta, df, claves, borrar_ausentes, col_borrado=None, cambio_tipo="ignorar"):
    """MERGE contra Delta. df ya viene deduplicado por claves y con _hash_fila.
    - col_borrado (opcional): columna booleana del origen; si es True, la fila se borra en destino.
    - borrar_ausentes: borra del destino lo que no viene en el origen (instantánea completa).
    - cambio_tipo: qué hacer si una columna existente cambia de tipo en el origen.
        "ignorar"    el MERGE convierte al tipo del destino (comportamiento histórico).
        "reescribir" reescribe la tabla con el esquema del origen (solo tiene sentido con borrar_ausentes).
        "fallar"     lanza un error.
    Devuelve (insertadas, actualizadas, borradas)."""
    cols = [c for c in df.columns if c != col_borrado]
    if not es_tabla_delta(ruta):
        datos = df.filter(~F.col(col_borrado)) if col_borrado else df
        datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
        return spark.read.format("delta").load(ruta).count(), 0, 0
    if cambio_tipo != "ignorar":
        cambiados = columnas_con_tipo_cambiado(ruta, df.select(*cols))
        if cambiados and cambio_tipo == "fallar":
            raise ValueError(f"Cambio de tipo en {cambiados} y la tabla no se puede reescribir (borrar_ausentes=False)")
        if cambiados:
            print(f"Cambio de tipo en {cambiados}: se reescribe la tabla con el esquema de la consulta")
            antes = spark.read.format("delta").load(ruta).count()
            datos = df.filter(~F.col(col_borrado)) if col_borrado else df
            datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
            return spark.read.format("delta").load(ruta).count(), 0, antes
    anadir_columnas_nuevas(ruta, df.select(*cols), evolucion_esquema)
    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in claves)
    asignar = {f"`{c}`": f"s.`{c}`" for c in cols}
    m = DeltaTable.forPath(spark, ruta).alias("t").merge(df.alias("s"), cond)
    if col_borrado:
        m = m.whenMatchedDelete(condition=f"s.`{col_borrado}`")
        m = m.whenMatchedUpdate(condition=f"NOT s.`{col_borrado}` AND NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(condition=f"NOT s.`{col_borrado}`", values=asignar)
    else:
        m = m.whenMatchedUpdate(condition="NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(values=asignar)
    if borrar_ausentes:
        m = m.whenNotMatchedBySourceDelete()
    m.execute()
    met = DeltaTable.forPath(spark, ruta).history(1).select("operationMetrics").first()[0] or {}
    return (int(met.get("numTargetRowsInserted", 0)), int(met.get("numTargetRowsUpdated", 0)),
            int(met.get("numTargetRowsDeleted", 0)))

In [ ]:
CAPA = "bronze_silver"
ID_ORQUESTACION = ID_EJECUCION
ACTIVIDADES = [
    {
        "dependencies": [],
        "name": "b2s_csv_cobros",
        "path": "b2s_csv_cobros"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_Department",
        "path": "b2s_HumanResources_Department"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_Employee",
        "path": "b2s_HumanResources_Employee"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_EmployeeDepartmentHistory",
        "path": "b2s_HumanResources_EmployeeDepartmentHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_EmployeePayHistory",
        "path": "b2s_HumanResources_EmployeePayHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_JobCandidate",
        "path": "b2s_HumanResources_JobCandidate"
    },
    {
        "dependencies": [],
        "name": "b2s_HumanResources_Shift",
        "path": "b2s_HumanResources_Shift"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_Address",
        "path": "b2s_Person_Address"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_AddressType",
        "path": "b2s_Person_AddressType"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_BusinessEntity",
        "path": "b2s_Person_BusinessEntity"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_BusinessEntityAddress",
        "path": "b2s_Person_BusinessEntityAddress"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_BusinessEntityContact",
        "path": "b2s_Person_BusinessEntityContact"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_ContactType",
        "path": "b2s_Person_ContactType"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_CountryRegion",
        "path": "b2s_Person_CountryRegion"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_EmailAddress",
        "path": "b2s_Person_EmailAddress"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_Password",
        "path": "b2s_Person_Password"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_Person",
        "path": "b2s_Person_Person"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_PersonPhone",
        "path": "b2s_Person_PersonPhone"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_PhoneNumberType",
        "path": "b2s_Person_PhoneNumberType"
    },
    {
        "dependencies": [],
        "name": "b2s_Person_StateProvince",
        "path": "b2s_Person_StateProvince"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_BillOfMaterials",
        "path": "b2s_Production_BillOfMaterials"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_Culture",
        "path": "b2s_Production_Culture"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_Document",
        "path": "b2s_Production_Document"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_Illustration",
        "path": "b2s_Production_Illustration"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_Location",
        "path": "b2s_Production_Location"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_Product",
        "path": "b2s_Production_Product"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductCategory",
        "path": "b2s_Production_ProductCategory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductCostHistory",
        "path": "b2s_Production_ProductCostHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductDescription",
        "path": "b2s_Production_ProductDescription"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductDocument",
        "path": "b2s_Production_ProductDocument"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductInventory",
        "path": "b2s_Production_ProductInventory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductListPriceHistory",
        "path": "b2s_Production_ProductListPriceHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductModel",
        "path": "b2s_Production_ProductModel"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductModelIllustration",
        "path": "b2s_Production_ProductModelIllustration"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductModelProductDescriptionCulture",
        "path": "b2s_Production_ProductModelProductDescriptionCulture"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductPhoto",
        "path": "b2s_Production_ProductPhoto"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductProductPhoto",
        "path": "b2s_Production_ProductProductPhoto"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductReview",
        "path": "b2s_Production_ProductReview"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ProductSubcategory",
        "path": "b2s_Production_ProductSubcategory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_ScrapReason",
        "path": "b2s_Production_ScrapReason"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_TransactionHistory",
        "path": "b2s_Production_TransactionHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_TransactionHistoryArchive",
        "path": "b2s_Production_TransactionHistoryArchive"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_UnitMeasure",
        "path": "b2s_Production_UnitMeasure"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_WorkOrder",
        "path": "b2s_Production_WorkOrder"
    },
    {
        "dependencies": [],
        "name": "b2s_Production_WorkOrderRouting",
        "path": "b2s_Production_WorkOrderRouting"
    },
    {
        "dependencies": [],
        "name": "b2s_Purchasing_ProductVendor",
        "path": "b2s_Purchasing_ProductVendor"
    },
    {
        "dependencies": [],
        "name": "b2s_Purchasing_PurchaseOrderDetail",
        "path": "b2s_Purchasing_PurchaseOrderDetail"
    },
    {
        "dependencies": [],
        "name": "b2s_Purchasing_PurchaseOrderHeader",
        "path": "b2s_Purchasing_PurchaseOrderHeader"
    },
    {
        "dependencies": [],
        "name": "b2s_Purchasing_ShipMethod",
        "path": "b2s_Purchasing_ShipMethod"
    },
    {
        "dependencies": [],
        "name": "b2s_Purchasing_Vendor",
        "path": "b2s_Purchasing_Vendor"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_CountryRegionCurrency",
        "path": "b2s_Sales_CountryRegionCurrency"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_CreditCard",
        "path": "b2s_Sales_CreditCard"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_Currency",
        "path": "b2s_Sales_Currency"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_CurrencyRate",
        "path": "b2s_Sales_CurrencyRate"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_Customer",
        "path": "b2s_Sales_Customer"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_PersonCreditCard",
        "path": "b2s_Sales_PersonCreditCard"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesOrderDetail",
        "path": "b2s_Sales_SalesOrderDetail"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesOrderHeader",
        "path": "b2s_Sales_SalesOrderHeader"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesOrderHeaderSalesReason",
        "path": "b2s_Sales_SalesOrderHeaderSalesReason"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesPerson",
        "path": "b2s_Sales_SalesPerson"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesPersonQuotaHistory",
        "path": "b2s_Sales_SalesPersonQuotaHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesReason",
        "path": "b2s_Sales_SalesReason"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesTaxRate",
        "path": "b2s_Sales_SalesTaxRate"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesTerritory",
        "path": "b2s_Sales_SalesTerritory"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SalesTerritoryHistory",
        "path": "b2s_Sales_SalesTerritoryHistory"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_ShoppingCartItem",
        "path": "b2s_Sales_ShoppingCartItem"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SpecialOffer",
        "path": "b2s_Sales_SpecialOffer"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_SpecialOfferProduct",
        "path": "b2s_Sales_SpecialOfferProduct"
    },
    {
        "dependencies": [],
        "name": "b2s_Sales_Store",
        "path": "b2s_Sales_Store"
    }
]

def con_dependencias(nombres, actividades):
    por_nombre = {a["name"]: a for a in actividades}
    pendientes, elegidas = list(nombres), set()
    while pendientes:
        n = pendientes.pop()
        if n not in por_nombre:
            raise ValueError(f"Actividad desconocida: {n}")
        if n not in elegidas:
            elegidas.add(n)
            pendientes.extend(por_nombre[n]["dependencies"])
    return [a for a in actividades if a["name"] in elegidas]

elegidas = con_dependencias([s.strip() for s in solo.split(",") if s.strip()], ACTIVIDADES) if solo.strip() else ACTIVIDADES
args = dict(id_orquestacion=ID_ORQUESTACION, simular=simular, modo=modo)
dag = {
    "activities": [dict(a, args=dict(args), timeoutPerCellInSeconds=timeout_celda_s, retry=reintentos,
                        retryIntervalInSeconds=30) for a in elegidas],
    "timeoutInSeconds": timeout_total_s,
    "concurrency": concurrencia or len(elegidas),
}

In [ ]:
estado, error, resultado = "OK", None, {}
try:
    resultado = notebookutils.notebook.runMultiple(dag, {"displayDAGViaGraphviz": False})
except Exception as e:
    estado, error = "ERROR", f"{type(e).__name__}: {e}"[:4000]

# Resumen desde el log de cada notebook (fuente única de verdad).
ok = sin_cambios = errores = 0
ruta_log = f"{RUTA_CONTROL}/log_ejecucion"
if es_tabla_delta(ruta_log):
    # Con reintentos puede haber varias filas por objeto: cuenta solo la última.
    ultima = Window.partitionBy("objeto").orderBy(F.col("fin").desc())
    filas = (spark.read.format("delta").load(ruta_log).filter(F.col("id_orquestacion") == ID_ORQUESTACION)
             .withColumn("_n", F.row_number().over(ultima)).filter("_n = 1")
             .groupBy("estado").count().collect())
    cuenta = {r["estado"]: r["count"] for r in filas}
    ok, sin_cambios, errores = cuenta.get("OK", 0), cuenta.get("SIN_CAMBIOS", 0), cuenta.get("ERROR", 0)
sin_log = len(elegidas) - ok - sin_cambios - errores   # notebooks que no llegaron a escribir (timeout, cancelados)
if errores or sin_log:
    estado = "ERROR"

fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
esquema = T.StructType([T.StructField(n, t) for n, t in [
    ("id_orquestacion", T.StringType()), ("proyecto", T.StringType()), ("capa", T.StringType()),
    ("estado", T.StringType()), ("inicio", T.TimestampType()), ("fin", T.TimestampType()),
    ("duracion_s", T.DoubleType()), ("actividades", T.LongType()), ("ok", T.LongType()),
    ("sin_cambios", T.LongType()), ("errores", T.LongType()), ("sin_log", T.LongType()),
    ("concurrencia", T.LongType()), ("parametros", T.StringType()), ("error", T.StringType()),
    ("notebook", T.StringType()), ("usuario", T.StringType()), ("spark_app_id", T.StringType())]])
fila = (ID_ORQUESTACION, proyecto, CAPA, estado, INICIO, fin, (fin - INICIO).total_seconds(), len(elegidas),
        ok, sin_cambios, errores, sin_log, dag["concurrency"], json.dumps(dict(args, solo=solo)), error,
        contexto("currentNotebookName"), contexto("userName"), spark.sparkContext.applicationId)
(spark.createDataFrame([fila], esquema).write.format("delta").mode("append").option("mergeSchema", "true")
     .save(f"{RUTA_CONTROL}/log_orquestacion"))
print(f"{CAPA}: {estado} · {ok} OK, {sin_cambios} sin cambios, {errores} con error, {sin_log} sin log · id_orquestacion={ID_ORQUESTACION}")
if estado == "ERROR":
    raise RuntimeError(error or f"{errores + sin_log} actividades con error. Consulta log_ejecucion con id_orquestacion={ID_ORQUESTACION}")

In [ ]:
notebookutils.notebook.exit(json.dumps(dict(estado=estado, id_orquestacion=ID_ORQUESTACION, ok=ok, sin_cambios=sin_cambios, errores=errores)))